In [1]:
import os
os.chdir(r"C:\Users\wangy")

In [2]:
import pandas as pd

# Raw diffusion file (no cluster)
raw = pd.read_excel("diffusion(S3)/diffusion_genderTopicvector.xlsx")
final_df = pd.read_excel("diffusion(S3)/final_results.xlsx")

# File that contains the TopContentCluster column
clusters = pd.read_excel("batch_outputs(S2)/filtered2_results_clustered_all.xlsx")

In [3]:
cluster_cols = clusters[['Title', 'TopContentCluster']].drop_duplicates()

In [4]:
raw = raw.merge(
    cluster_cols,
    left_on="article_title",
    right_on="Title",
    how="left",
    suffixes=("", "_cluster")
)

# drop safely (won't error if not there)
raw = raw.drop(columns=["Title", "Title_cluster"], errors="ignore")

# ---- merge into final results ----
final_df = final_df.merge(
    cluster_cols,
    left_on="article_title",
    right_on="Title",
    how="left",
    suffixes=("", "_cluster")
)

final_df = final_df.drop(columns=["Title", "Title_cluster"], errors="ignore")

In [5]:
# Step 0 — Cascade size per message
cascade_size = (
    raw.groupby(["agent_name", "article_title", "TopContentCluster"])
        .size()
        .rename("cascade_size")
        .reset_index()
)

# Merge cascade size into final_df
final_df = final_df.merge(
    cascade_size,
    on=["agent_name", "article_title", "TopContentCluster"],
    how="left"
)

In [6]:
g = final_df.groupby(["agent_name", "TopContentCluster"])

topic_agent_agg = g.agg(
    # 1) 1st degree width max, avg
    first_layer_width_max=("first_layer_width", "max"),
    first_layer_width_avg=("first_layer_width", "mean"),

    # 2) 2nd degree width max, avg
    second_layer_width_max=("second_layer_width", "max"),
    second_layer_width_avg=("second_layer_width", "mean"),

    # 3) Cascade size mean/var
    cascade_size_mean=("cascade_size", "mean"),
    cascade_size_var=("cascade_size", "var"),

    # 4) Depth mean/var
    depth_mean=("depth", "mean"),
    depth_var=("depth", "var"),

    # 7) Reshare stats
    reshare_mean=("reshare_pct", "mean"),
    reshare_var=("reshare_pct", "var"),

    # 8–10 other metrics
    structural_virality_mean=("structural_virality", "mean"),
    centrality_mean=("centrality", "mean"),
    wiener_index_mean=("wiener_index", "mean"),
    agent_deg_centrality_mean=("agent_deg_centrality", "mean"),
    avg_out_degree_centrality_mean=("avg_out_degree_centrality", "mean"),

    # 11) gender %
    gender_pct_all_1_mean=("gender_pct_all_1", "mean"),
    gender_pct_all_0_mean=("gender_pct_all_0", "mean"),

    # 12) gender homophily
    gender_assortativity_mean=("gender_assortativity", "mean"),

    # 13) duration
    duration_mean_of_means=("duration_mean_s", "mean"),
    duration_var_of_means=("duration_mean_s", "var"),
    duration_mean_of_vars=("duration_var_s2", "mean"),
    duration_var_of_vars=("duration_var_s2", "var"),
).reset_index()

In [9]:
import numpy as np

READER_COL = "reader_wechat_nn"

KEYS = ["agent_name", "TopContentCluster"]

def repeat_nodes_agent_topic(raw: pd.DataFrame, layer: int, out_col: str):
    # 1) node set: readers who appear in this layer at least once within (agent, topic)
    layer_nodes = (
        raw[raw["correct_layer"] == layer][KEYS + [READER_COL]]
        .drop_duplicates()
    )

    # 2) reader-article membership across ANY layer, within (agent, topic)
    raw_any = raw[KEYS + ["article_title", READER_COL]].drop_duplicates()

    # 3) total articles per (agent, topic)
    total_articles = (
        raw_any.groupby(KEYS)["article_title"]
               .nunique()
               .rename("total_articles_any")
               .reset_index()
    )

    # 4) articles seen per reader per (agent, topic)
    articles_seen = (
        raw_any.groupby(KEYS + [READER_COL])["article_title"]
               .nunique()
               .rename("articles_seen_any")
               .reset_index()
               .merge(total_articles, on=KEYS, how="left")
    )

    # 5) keep only layer-defined nodes
    tmp = layer_nodes.merge(articles_seen, on=KEYS + [READER_COL], how="left")

    # 6) repeat node
    tmp["is_repeat"] = tmp["articles_seen_any"] >= np.ceil(0.3 * tmp["total_articles_any"])

    # 7) % repeat nodes per (agent, topic)
    out = (
        tmp.groupby(KEYS)["is_repeat"]
           .mean()
           .rename(out_col)
           .reset_index()
    )
    return out


# compute 1st + 2nd degree agent-topic repeat exposure ----
repeat_1st_agent_topic = repeat_nodes_agent_topic(
    raw=raw, layer=1, out_col="repeat_exposure_1st_nodes_pct"
)

repeat_2nd_agent_topic = repeat_nodes_agent_topic(
    raw=raw, layer=2, out_col="repeat_exposure_2nd_nodes_pct"
)

# join into agent_topic_agg (must have agent_name + TopContentCluster) ----
agent_topic_agg = topic_agent_agg.merge(
    repeat_1st_agent_topic, on=KEYS, how="left"
).merge(
    repeat_2nd_agent_topic, on=KEYS, how="left"
)


In [12]:
topic_agent_agg = agent_topic_agg.fillna(0)
topic_agent_agg.head()

,agent_name,TopContentCluster,first_layer_width_max,first_layer_width_avg,second_layer_width_max,second_layer_width_avg,cascade_size_mean,cascade_size_var,depth_mean,depth_var,...,avg_out_degree_centrality_mean,gender_pct_all_1_mean,gender_pct_all_0_mean,gender_assortativity_mean,duration_mean_of_means,duration_var_of_means,duration_mean_of_vars,duration_var_of_vars,repeat_exposure_1st_nodes_pct,repeat_exposure_2nd_nodes_pct
0,+章珊,家居设计与装修,21,4.5,1,0.038462,6.076923,22.713846,0.884615,0.186154,...,0.116630,0.328926,0.671074,-0.099246,603.209003,3.845653e+05,1.951193e+06,1.067840e+13,0.017857,0.0
1,+章珊,活动与促销,7,3.5,0,0.000000,4.000000,18.000000,0.500000,0.500000,...,0.062500,0.428571,0.571429,-0.123288,2185.714286,6.951122e+06,1.157143e+05,0.000000e+00,1.000000,0.0
2,+章珊,生活方式与文化,4,3.5,5,2.500000,7.500000,24.500000,1.500000,0.500000,...,0.115909,0.431818,0.568182,-0.095442,675.000000,9.112500e+05,9.922500e+05,1.969120e+12,1.000000,1.0
3,丁丽美,家居设计与装修,13,9.0,0,0.000000,9.333333,30.333333,1.000000,0.000000,...,0.130647,0.380342,0.619658,-0.062515,779.807692,5.275213e+05,6.496796e+06,1.067345e+14,1.000000,0.0
4,丁丽美,活动与促销,5,3.5,0,0.000000,4.000000,8.000000,1.000000,0.000000,...,0.226190,0.166667,0.833333,-0.030303,412.500000,3.403125e+05,1.272375e+06,3.237876e+12,1.000000,0.0


In [13]:
topic_agent_agg.to_excel("diffusion(S3)/agent_topic_level_results.xlsx", index=False)